# Reproducing the paper: all 8 algorithms
**Group 17** · BITS Pilani WILP · DRL Assignment 1
Paper: *A Framework for Fair Evaluation of Variance-Aware Bandit Algorithms*, arXiv:2510.27001

### What this does
The paper's claims are measurements, not proofs, so we re-run its experiment. We simulate
three two-armed Bernoulli bandits for a million steps each and measure **regret** — how much
reward an algorithm loses compared with always playing the best arm. Lower is better.

### The one idea being tested
Six of the eight algorithms score each arm as `empirical mean + bonus` and pull the highest
score. Only the bonus differs, and that bonus is the entire subject of the paper:

| Algorithm | Bonus | Idea |
|---|---|---|
| `UCB` | `√(2 log t / n)` | explore an arm less as you pull it more |
| `UCB-V` | `√(2·V·log t / n) + 3 log t / n` | **also** explore more when the arm is noisy |
| `UCB-Tuned` | `√(log t / n · min(¼, V + √(2 log t / n)))` | same, variance capped at ¼ |
| `EUCBV` | variance bound **+ elimination** | drop arms that cannot win |
| `PAC-UCB` | `√(p/n) + p/n`, `p = log(Kt^1.3/β)` | no variance term |
| `UCB-Improved` | uniform sampling in rounds **+ elimination** | halving accuracy target |

`V` is the arm's measured variance. The remaining two use no bonus at all: `ETC` explores a
fixed number of pulls then commits forever, and `eps-Greedy` picks at random 1% of the time.

### 1. The simulation
One loop: pick an arm, get a 0 or 1, update beliefs, add up regret.

In [1]:
import math, random, time

# ----------------------------------------------------------------- simulation
def run(algo, probs, T, seed, m=100, eps=0.01, rho=0.5):
    """Plays one bandit for T steps and returns the total regret.

    Every algorithm scores each arm as (empirical mean + bonus) and pulls the
    highest score. Only the bonus differs -- that bonus is the whole paper.
    ETC and eps-Greedy are the two that do not use a bonus at all.
    """
    rng = random.Random(seed)
    K = len(probs)
    n = [0] * K            # pulls per arm
    mean = [0.0] * K       # empirical mean per arm
    m2 = [0.0] * K         # running sum for the empirical variance
    active = list(range(K))   # arms not yet eliminated (EUCBV, UCB-Improved)
    committed = None          # the arm ETC settles on
    delta = 1.0               # UCB-Improved's accuracy estimate, halved each round
    log_pac = math.log(K / 0.05)   # precomputed constant for PAC-UCB
    best_p = max(probs)
    regret = 0.0

    for t in range(1, T + 1):
        # ---- 1. choose an arm ---------------------------------------------
        if t <= K:
            a = t - 1                                    # one free pull each

        elif algo == "ETC":                              # explore, then commit
            if t <= m * K:
                a = (t - 1) % K
            else:
                if committed is None:
                    committed = mean.index(max(mean))
                a = committed

        elif algo == "eps-Greedy":                       # random with prob eps
            a = rng.randrange(K) if rng.random() < eps else mean.index(max(mean))

        elif algo == "UCB-Improved":
            # Samples the surviving arms UNIFORMLY within a round, not by index.
            # Selecting by UCB instead starves the worse arm, its pull count
            # never reaches the round target, and elimination never fires.
            a = min(active, key=lambda i: n[i])

        else:                                            # the six UCB variants
            lt = math.log(t)
            best_score, a = -1e18, active[0]
            for i in active:
                v = m2[i] / n[i] if n[i] > 1 else 0.0    # empirical variance
                if algo == "UCB":
                    bonus = math.sqrt(2 * lt / n[i])
                elif algo == "UCB-V":
                    bonus = math.sqrt(2 * v * lt / n[i]) + 3 * lt / n[i]
                elif algo == "UCB-Tuned":
                    cap = min(0.25, v + math.sqrt(2 * lt / n[i]))
                    bonus = math.sqrt(lt / n[i] * cap)
                elif algo == "EUCBV":
                    e = max(math.log(rho * T / t), 1.0)
                    bonus = math.sqrt(v * e / n[i]) + e / n[i]
                else:                                    # PAC-UCB
                    p = log_pac + 1.3 * lt
                    bonus = math.sqrt(p / n[i]) + p / n[i]
                if mean[i] + bonus > best_score:
                    best_score, a = mean[i] + bonus, i

        # ---- 2. pull it, update beliefs, add up regret ---------------------
        r = 1 if rng.random() < probs[a] else 0
        n[a] += 1
        d = r - mean[a]
        mean[a] += d / n[a]
        m2[a] += d * (r - mean[a])
        regret += best_p - probs[a]

        # ---- 3. the two eliminating algorithms drop hopeless arms ----------
        # An arm is dropped when its upper bound sits below the best arm's
        # LOWER bound, i.e. it cannot still be the winner.
        if algo == "EUCBV" and len(active) > 1 and all(n[i] >= 2 for i in active):
            e = max(math.log(rho * T / t), 1.0)
            width = [math.sqrt((m2[i] / n[i]) * e / n[i]) + e / n[i]
                     for i in range(K)]
            best_lb = max(mean[i] - width[i] for i in active)
            keep = [i for i in active if mean[i] + width[i] >= best_lb]
            if keep:
                active = keep

        elif algo == "UCB-Improved" and len(active) > 1:
            # Rounds of a halving accuracy delta. The elimination width must be
            # TIGHTER than the selection bonus, or no arm is ever dropped and
            # the algorithm silently degenerates into plain UCB.
            logd = math.log(max(T * delta * delta, math.e))
            n_round = math.ceil(2 * logd / (delta * delta))
            if all(n[i] >= n_round for i in active):
                c = math.sqrt(logd / (2 * n_round))
                best_lb = max(mean[i] - c for i in active)
                keep = [i for i in active if mean[i] + c >= best_lb]
                if keep:
                    active = keep
                delta /= 2

    return regret

### 2. Run all eight on the paper's three scenarios

| Scenario | Arms | Gap |
|---|---|---|
| A | 0.8 vs 0.9 | 0.100 — easy |
| B | 0.895 vs 0.9 | 0.005 — hard |
| C | 0.89 vs 0.895 | 0.005 — hard and noisier |

Both arm orderings are run, so `argmax` ties cannot favour arm 0. Takes about 3 minutes.

In [2]:
# ------------------------------------------------------------------ the 8 algos
# (label, algo key, extra settings)
ALGOS = [
    ("ETC(m=100)",    "ETC",          dict(m=100)),
    ("ETC(m=10000)",  "ETC",          dict(m=10_000)),
    ("eps-Greedy",    "eps-Greedy",   dict(eps=0.01)),
    ("UCB",           "UCB",          {}),
    ("UCB-V",         "UCB-V",        {}),
    ("UCB-Tuned",     "UCB-Tuned",    {}),
    ("EUCBV",         "EUCBV",        {}),
    ("UCB-Improved",  "UCB-Improved", {}),
]

SCENARIOS = {"A": [0.8, 0.9], "B": [0.895, 0.9], "C": [0.89, 0.895]}

# Figures the paper reports, for the cells where it reports one.
PAPER = {
    ("A", "UCB"): 238.17,  ("A", "UCB-V"): 109.93, ("A", "UCB-Tuned"): 30.67,
    ("A", "EUCBV"): 47.73, ("A", "ETC(m=100)"): 10.10,
    ("A", "UCB-Improved"): 32500.0,
    ("B", "UCB"): 1127.17, ("B", "UCB-Tuned"): 212.21,
    ("B", "ETC(m=10000)"): 150.66,
    ("C", "UCB"): 1172.57, ("C", "UCB-Tuned"): 226.09,
}

T, TRIALS = 1_000_000, 4
t0 = time.time()
print(f"{'Scen':6s}{'Algorithm':15s}{'Our mean':>11s}{'Our median':>12s}"
      f"{'Paper':>10s}{'Diff':>8s}")
print("-" * 62)
for s, probs in SCENARIOS.items():
    for label, key, kw in ALGOS:
        runs = []
        for order in (probs, probs[::-1]):            # both arm orderings
            for k in range(TRIALS):
                runs.append(run(key, order, T, seed=1000 + k, **kw))
        runs.sort()
        mean_r = sum(runs) / len(runs)
        med_r = runs[len(runs) // 2]
        pv = PAPER.get((s, label))
        mark = " †" if label in ("EUCBV", "UCB-Improved") else ""
        diff = f"{(mean_r - pv) / pv * 100:+6.0f}%" if pv else f"{'-':>7s}"
        pv_s = f"{pv:10.2f}" if pv else f"{'-':>10s}"
        print(f"{s:6s}{(label + mark):15s}{mean_r:11.2f}{med_r:12.2f}{pv_s}{diff}")
    print()
print(f"T = {T:,} | {TRIALS * 2} runs per cell | {time.time() - t0:.0f}s")
print("† simplified re-implementation; divergence from the paper is a limit of "
      "our code, not a finding about the paper.")

Scen  Algorithm         Our mean  Our median     Paper    Diff
--------------------------------------------------------------


A     ETC(m=100)           10.00       10.00     10.10    -1%


A     ETC(m=10000)       1000.00     1000.00         -      -


A     eps-Greedy          544.13      544.10         -      -


A     UCB                 238.05      257.40    238.17    -0%


...

C     UCB                1097.91     1120.61   1172.57    -6%


C     UCB-V               308.58      296.77         -      -


C     UCB-Tuned           240.44      268.48    226.09    +6%


C     EUCBV †              48.07       22.70         -      -


C     UCB-Improved †     1964.47     1785.96         -      -

T = 1,000,000 | 8 runs per cell | 164s
† simplified re-implementation; divergence from the paper is a limit of our code, not a finding about the paper.


### What the result shows

**1. The reproduction works.** `UCB` lands within 6% of the paper in all three scenarios
(238.05 vs 238.17 in A). `UCB-V` is within 8%, `UCB-Tuned` within 6–31%.

**2. Variance-aware exploration pays off when the arms are close.** In Scenario C,
`UCB-Tuned` loses 240 where plain `UCB` loses 1098 — about **4.5× less**. In the easy
Scenario A the advantage is much smaller. That is the paper's central claim, measured.

**3. ETC's reported win does not survive repetition.** Look at the gap between its mean and
its median in Scenarios B and C: mean 2500, median 4999. `ETC` commits permanently after
exploring, so if it commits to the wrong arm it pays for the rest of the horizon. Roughly
half our runs did exactly that. A mean over such a split distribution describes neither
outcome, which matters because the paper argues for distributional metrics and
Value-at-Risk precisely on the grounds that means hide tail behaviour — and then ranks the
algorithms by the mean.

**4. Honest limitation.** The two rows marked † are simplified re-implementations. Our
`EUCBV` eliminates more aggressively than the paper's and our `UCB-Improved` far less, so
those two numbers do not reproduce. That is a limitation of our code, not a finding about
the paper.